In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df1 = pd.read_csv('../data/raw/weight_change_dataset.csv')
df2 = pd.read_csv('../data/raw/obesity_dataset.csv')
print("Основной датасет:", df1.shape)
print("UCI датасет:", df2.shape)

df1['Current Weight (kg)'] = df1['Current Weight (lbs)'] * 0.453592
df1['Final Weight (kg)'] = df1['Final Weight (lbs)'] * 0.453592
df1['Weight Change (kg)'] = df1['Weight Change (lbs)'] * 0.453592
df1.head()

#доля дефицита/избытка ккал относительно bmr
df1['Deficit_Ratio'] = df1['Daily Caloric Surplus/Deficit'] / df1['BMR (Calories)']

#изменение веса в неделю
df1['Weight_Change_per_Week'] = df1['Weight Change (lbs)'] / df1['Duration (weeks)']
df1[['Deficit_Ratio', 'Weight_Change_per_Week']].describe()

Q1 = df1['Weight Change (lbs)'].quantile(0.25)
Q3 = df1['Weight Change (lbs)'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df1['Is_Outlier'] = ((df1['Weight Change (lbs)'] < lower) |(df1['Weight Change (lbs)'] > upper)).astype(int)
print("Выбросов найдено:", df1['Is_Outlier'].sum())

le_gender = LabelEncoder()
df1['Gender_encoded'] = le_gender.fit_transform(df1['Gender'])

activity_order = {'Sedentary': 0, 'Lightly Active': 1,'Moderately Active': 2, 'Very Active': 3}
df1['Activity_encoded'] = df1['Physical Activity Level'].map(activity_order)
sleep_order = {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3}
df1['Sleep_encoded'] = df1['Sleep Quality'].map(sleep_order)
df1[['Gender_encoded', 'Activity_encoded', 'Sleep_encoded']].head()



df2['BMI'] = df2['Weight'] / (df2['Height'] ** 2)
le_gender2 = LabelEncoder()
df2['Gender_encoded'] = le_gender2.fit_transform(df2['Gender'])
binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    df2[col + '_encoded'] = df2[col].map({'yes': 1, 'no': 0})

le_target = LabelEncoder()
df2['NObeyesdad_encoded'] = le_target.fit_transform(df2['NObeyesdad'])
df2.head()
scaler1 = StandardScaler()
numeric_features_1 = ['Age', 'Current Weight (kg)', 'Daily Calories Consumed', 'Daily Caloric Surplus/Deficit', 'Duration (weeks)', 'Stress Level']
df1_scaled = df1.copy()
df1_scaled[numeric_features_1] = scaler1.fit_transform(df1[numeric_features_1])
scaler2 = StandardScaler()
numeric_features_2 = ['Age', 'Height', 'Weight', 'BMI', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
df2_scaled = df2.copy()
df2_scaled[numeric_features_2] = scaler2.fit_transform(df2[numeric_features_2])

#предсказываем Weight Change
X1 = df1_scaled[numeric_features_1 + ['Gender_encoded', 'Activity_encoded', 'Sleep_encoded']]
y1 = df1_scaled['Weight Change (lbs)']
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

#предсказываем категорию веса
X2 = df2_scaled[numeric_features_2 + ['Gender_encoded'] + [c + '_encoded' for c in binary_cols]]
y2 = df2_scaled['NObeyesdad_encoded']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)
print("Основной датасет — train/test:", X1_train.shape, X1_test.shape)
print("UCI датасет — train/test:", X2_train.shape, X2_test.shape)

df1_scaled.to_csv('../data/processed/weight_change_processed.csv', index=False)
df2_scaled.to_csv('../data/processed/obesity_processed.csv', index=False)
X1_train.to_csv('../data/processed/X1_train.csv', index=False)
X1_test.to_csv('../data/processed/X1_test.csv', index=False)
y1_train.to_csv('../data/processed/y1_train.csv', index=False)
y1_test.to_csv('../data/processed/y1_test.csv', index=False)
print("Готово:  обработанные данные сохранены")

Основной датасет: (100, 13)
UCI датасет: (2087, 17)
Выбросов найдено: 6
Основной датасет — train/test: (80, 9) (20, 9)
UCI датасет — train/test: (1669, 14) (418, 14)
Готово:  обработанные данные сохранены


## Выводы по предобработке

1) Основной датасет: добавлены веса в кг, признак недельного изменения веса, доля дефицита от BMR, помечены выбросы(не удалены), закодированы категориальные признаки
2) UCI датасет: рассчитан BMI, закодированы бинарные и категориальные признаки, целевая переменная(7 категорий) закодирована
3) Оба датасета масштабированы(StandardScaler) и разбиты на train/test(80/20)
4) Данные готовы для передачи на этап моделирования